In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone (UTC+8) variable as required (not used to convert timestamps here)
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Ensure timestamp and value types
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

for df_name, df in (('metric', metric_df), ('trace', trace_df), ('log', log_df)):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) metric.csv aggregations
# basic aggregates
metric_agg = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('timestamp', 'size'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max'),
    max_value=('value', 'max')
).reset_index()

# global P50 and P95 per (cmdb_id, kpi_name)
metric_q = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile([0.5, 0.95]).unstack(level=-1)
metric_q = metric_q.rename(columns={0.5: 'p50', 0.95: 'p95'}).reset_index()

metric_agg = metric_agg.merge(metric_q, on=['cmdb_id', 'kpi_name'], how='left')

# sort by count desc and take top 20
metric_agg_top20 = metric_agg.sort_values('count', ascending=False).head(20).loc[
    :, ['cmdb_id', 'kpi_name', 'count', 'min_ts', 'max_ts', 'p50', 'p95', 'max_value']
].reset_index(drop=True)

# overall metric timestamps
metric_overall_ts = {
    'overall_min_timestamp': int(metric_df['timestamp'].min()) if metric_df['timestamp'].notna().any() else pd.NA,
    'overall_max_timestamp': int(metric_df['timestamp'].max()) if metric_df['timestamp'].notna().any() else pd.NA
}
metric_overall_ts_df = pd.DataFrame([metric_overall_ts])

# 2) trace.csv aggregations
trace_agg = trace_df.groupby(['cmdb_id', 'trace_name']).agg(
    count=('timestamp', 'size'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max'),
    max_value=('value', 'max')
).reset_index()

trace_q = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].quantile([0.5, 0.95]).unstack(level=-1)
trace_q = trace_q.rename(columns={0.5: 'p50', 0.95: 'p95'}).reset_index()

trace_agg = trace_agg.merge(trace_q, on=['cmdb_id', 'trace_name'], how='left')

trace_agg_top20 = trace_agg.sort_values('count', ascending=False).head(20).loc[
    :, ['cmdb_id', 'trace_name', 'count', 'min_ts', 'max_ts', 'p50', 'p95', 'max_value']
].reset_index(drop=True)

# 3) log.csv aggregations
log_agg = log_df.groupby(['cmdb_id', 'log_name']).agg(
    count=('timestamp', 'size'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max'),
    max_value=('value', 'max')
).reset_index()

log_q = log_df.groupby(['cmdb_id', 'log_name'])['value'].quantile([0.5, 0.95]).unstack(level=-1)
log_q = log_q.rename(columns={0.5: 'p50', 0.95: 'p95'}).reset_index()

log_agg = log_agg.merge(log_q, on=['cmdb_id', 'log_name'], how='left')

log_agg_top20 = log_agg.sort_values('count', ascending=False).head(20).loc[
    :, ['cmdb_id', 'log_name', 'count', 'min_ts', 'max_ts', 'p50', 'p95', 'max_value']
].reset_index(drop=True)

# 4) error_logs.csv aggregations
error_agg = error_logs_df.groupby('cmdb_id').agg(
    count=('timestamp', 'size'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max')
).reset_index().sort_values('count', ascending=False).reset_index(drop=True)

# up to 5 sample messages (earliest)
sample_messages = error_logs_df.sort_values('timestamp').head(5)[['timestamp', 'cmdb_id', 'message']].reset_index(drop=True)

# Final displayed outputs (compact)
metric_agg_top20, metric_overall_ts_df, trace_agg_top20, log_agg_top20, error_agg, sample_messages

```
Out[1]:
```
```python
# Reuse previously computed aggregation variables from the kernel:
# metric_agg_top20, metric_overall_ts_df, trace_agg_top20, log_agg_top20, error_agg, sample_messages

# Build a concise plain-English summary string
parts = []

# Metric summary
try:
    overall_min = int(metric_overall_ts_df.at[0, 'overall_min_timestamp'])
    overall_max = int(metric_overall_ts_df.at[0, 'overall_max_timestamp'])
except Exception:
    overall_min = overall_max = 'N/A'

metric_top_sample = []
for i in range(min(4, len(metric_agg_top20))):
    row = metric_agg_top20.iloc[i]
    metric_top_sample.append(
        f"{row['cmdb_id']} | {row['kpi_name']} (count={int(row['count'])}, p50={row['p50']:.6g}, p95={row['p95']:.6g}, max={row['max_value']:.6g})"
    )

parts.append(
    "Metrics: Top series mostly have count=30. Overall metric timestamp range: "
    f"{overall_min} to {overall_max}. Examples: " + "; ".join(metric_top_sample) + "."
)

# Trace summary
trace_top_sample = []
for i in range(min(4, len(trace_agg_top20))):
    row = trace_agg_top20.iloc[i]
    trace_top_sample.append(
        f"{row['cmdb_id']} | {row['trace_name']} (count={int(row['count'])}, p50={row['p50']:.6g}, p95={row['p95']:.6g})"
    )

parts.append(
    "Traces: Top trace series are from IG01/IG02 with count=30. Examples: " + "; ".join(trace_top_sample) + "."
)

# Log summary
log_top_sample = []
for i in range(min(6, len(log_agg_top20))):
    row = log_agg_top20.iloc[i]
    log_top_sample.append(
        f"{row['cmdb_id']} | {row['log_name']} (count={int(row['count'])}, p50={row['p50']:.6g}, p95={row['p95']:.6g})"
    )

parts.append(
    "Logs: Common series have count=26 (Tomcat and Apache). Examples: " + "; ".join(log_top_sample) + "."
)

# Error logs summary
if error_agg.shape[0] == 0:
    err_summary = "No error_logs entries found (aggregations are empty)."
else:
    err_rows = []
    for i in range(min(10, error_agg.shape[0])):
        r = error_agg.iloc[i]
        err_rows.append(f"{r['cmdb_id']} (count={int(r['count'])}, min_ts={int(r['min_ts'])}, max_ts={int(r['max_ts'])})")
    err_sample_msg = sample_messages.to_dict(orient='records') if not sample_messages.empty else []
    err_summary = "Error logs per cmdb_id: " + "; ".join(err_rows) + ". Sample messages (up to 5 earliest): " + str(err_sample_msg)

parts.append(err_summary)

summary = " ".join(parts)

# Display the concise summary string as the final output
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count      min_ts      max_ts           p50           p95     max_value
0    Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     30  1615563000  1615564740  0.000000e+00  0.000000e+00  0.000000e+00
1   apache02                        OSLinux-CPU_CPU_CPUidleutil     30  1615563000  1615564740  9.953260e+01  9.969062e+01  9.972156e+01
2       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     30  1615563000  1615564740  7.050000e+01  7.400000e+01  7.500000e+01
3       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     30  1615563000  1615564740  2.040136e+09  2.040136e+09  2.040136e+09
4       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     30  1615563000  1615564740  3.877292e+08  8.513503e+08  8.708300e+08
5       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     30  1615563000  1615564740  5.796410e+07  5.796410e+07  5.796410e+07
6   apache02                             OSLinux-CPU_CPU_CPUWio     30  1615563000  1615564740  2.080000e-02  1.989900e-01  6.681900e+00
7   apache02                        OSLinux-CPU_CPU_CPUUserTime     30  1615563000  1615564740  1.309000e-01  2.883150e-01  2.908800e+00
8   apache02                         OSLinux-CPU_CPU_CPUSysTime     30  1615563000  1615564740  2.908715e-01  3.853400e-01  4.820000e-01
9    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     30  1615563000  1615564740  0.000000e+00  0.000000e+00  0.000000e+00
10   Redis01                      redis-Redis_6379_redis server     30  1615563000  1615564740  1.000000e+00  1.000000e+00  1.000000e+00
11   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     30  1615563000  1615564740  4.000000e-01  4.000000e-01  4.000000e-01
12   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     30  1615563000  1615564740  7.000000e+00  7.000000e+00  7.000000e+00
13   Redis01     redis-Redis_6379_Redis  (rejected_connections)     30  1615563000  1615564740  0.000000e+00  0.000000e+00  0.000000e+00
14   Redis01  redis-Redis_6379_Redis  (total_commands_proces...     30  1615563000  1615564740  5.567451e+09  5.567952e+09  5.568009e+09
15  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     30  1615563000  1615564740  0.000000e+00  0.000000e+00  0.000000e+00
16  apache02                            OSLinux-CPU_CPU_CPULoad     30  1615563000  1615564740  2.500000e-02  1.465000e-01  2.300000e-01
17  apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     30  1615563000  1615564740  0.000000e+00  1.100000e+00  3.000000e+00
18  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     30  1615563000  1615564740  0.000000e+00  0.000000e+00  0.000000e+00
19   Mysql01                         OSLinux-CPU_CPU_CPUSysTime     30  1615563000  1615564740  3.303045e-01  4.651744e-01  5.318490e-01,    overall_min_timestamp  overall_max_timestamp
0             1615563000             1615564740,    cmdb_id                       trace_name  count      min_ts      max_ts         p50         p95   max_value
0     IG01             trace.self.row_count     30  1615563000  1615564740  303.000000  406.100000  596.000000
1     IG01          trace.self.duration_p95     30  1615563000  1615564740    0.358425    1.066005    1.691000
2     IG01         trace.self.duration_mean     30  1615563000  1615564740    0.082419    0.185535    0.296293
3     IG01  trace.to_Tomcat01.duration_mean     30  1615563000  1615564740    0.081791    0.254133    0.270600
4     IG01  trace.to_Tomcat02.duration_mean     30  1615563000  1615564740    0.085986    0.176123    0.223068
5     IG01      trace.to_Tomcat01.row_count     30  1615563000  1615564740   37.000000   59.200000   65.000000
6     IG01   trace.to_Tomcat01.duration_p95     30  1615563000  1615564740    0.290725    1.174050    1.581400
7     IG01      trace.to_Tomcat02.row_count     30  1615563000  1615564740   37.000000   55.300000   82.000000
8     IG01  trace.to_Tomcat03.duration_mean     30  1615563000  1615564740    0.077958    0.150452    0.249673
9     IG01   trace.to_Tomcat03.duration_p95     30  1615563000  1615564740    0.348225    0.895770    1.022600
10    IG01   trace.to_Tomcat02.duration_p95     30  1615563000  1615564740    0.397000    0.998562    1.088400
11    IG01      trace.to_Tomcat03.row_count     30  1615563000  1615564740   37.500000   49.550000   97.000000
12    IG01      trace.to_Tomcat04.row_count     30  1615563000  1615564740   38.000000   49.550000   54.000000
13    IG01   trace.to_Tomcat04.duration_p95     30  1615563000  1615564740    0.349500    1.090625    1.133750
14    IG01  trace.to_Tomcat04.duration_mean     30  1615563000  1615564740    0.077491    0.202408    0.224415
15    IG02  trace.to_Tomcat02.duration_mean     30  1615563000  1615564740    0.083550    0.180188    0.236860
16    IG02   trace.to_Tomcat02.duration_p95     30  1615563000  1615564740    0.351200    1.002745    1.081400
17    IG02      trace.to_Tomcat02.row_count     30  1615563000  1615564740   38.000000   50.550000   67.000000
18    IG02          trace.self.duration_p95     30  1615563000  1615564740    0.425525    0.987752    1.954000
19    IG02         trace.self.duration_mean     30  1615563000  1615564740    0.086223    0.146907    0.343475,      cmdb_id          log_name  count      min_ts      max_ts      p50       p95  max_value
0   Tomcat01     log.row_count     26  1615563000  1615564500   72.500   90.0000     109.00
1   Tomcat01   log.error_count     26  1615563000  1615564500    0.000    0.0000       0.00
2   apache02   log.error_count     26  1615563000  1615564500    0.000    0.0000       0.00
3   apache01     log.row_count     26  1615563000  1615564500  146.000  173.7500     183.00
4   Tomcat02   log.error_count     26  1615563000  1615564500    0.000    0.0000       0.00
5   Tomcat02     log.row_count     26  1615563000  1615564500   71.500   95.2500     102.00
6   Tomcat03   log.error_count     26  1615563000  1615564500    0.000    0.0000       0.00
7   Tomcat04     log.row_count     26  1615563000  1615564500   71.000   93.5000      98.00
8   Tomcat04   log.error_count     26  1615563000  1615564500    0.000    0.0000       0.00
9   Tomcat03     log.row_count     26  1615563000  1615564500   72.000   89.5000      93.00
10  apache02     log.row_count     26  1615563000  1615564500  146.500  178.0000     190.00
11  apache01   log.error_count     26  1615563000  1615564500    0.000    0.0000       0.00
12   Mysql02     log.row_count     20  1615563060  1615564500    7.500   94.6500     126.00
13   Mysql02   log.error_count     20  1615563060  1615564500    0.000    0.0000       0.00
14  Tomcat02  log.gc_pause_max      7  1615563000  1615564380    0.060    0.0840       0.09
15  Tomcat03  log.gc_pause_max      5  1615563000  1615564320    0.050    0.1160       0.13
16  Tomcat01  log.gc_pause_max      4  1615563240  1615564320    0.065    0.0785       0.08
17  Tomcat04  log.gc_pause_max      4  1615563180  1615564320    0.075    0.1155       0.12, Empty DataFrame
Columns: [cmdb_id, count, min_ts, max_ts]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Define allowed components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# 1) Calculate global P95 per series using the entire metric_df (rule 9)
global_p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
global_p95_df = global_p95_df.rename(columns={'value': 'global_p95'})

# Restrict to allowed components
global_p95_df = global_p95_df[global_p95_df['cmdb_id'].isin(allowed_components)].reset_index(drop=True)

# 2) Define incident window in UTC+8 and convert to epoch seconds
start_ts = int(pd.Timestamp('2021-03-12 23:30:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-13 00:00:00', tz=tz).timestamp())

# 3) Extract windowed data for allowed components
window_df = metric_df[
    (metric_df['cmdb_id'].isin(allowed_components)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Merge global_p95 into windowed rows
window_with_p95 = window_df.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Identify anomaly points where value >= global_p95
# (rows with missing global_p95 will be excluded)
anomalies_df = window_with_p95[
    window_with_p95['global_p95'].notna() &
    (window_with_p95['value'] >= window_with_p95['global_p95'])
].copy()

# 5) Aggregate anomalies per series
anomaly_summary = anomalies_df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    anomaly_count=('timestamp', 'size'),
    earliest_anomaly_ts=('timestamp', 'min'),
    max_value_in_window=('value', 'max')
)

# attach global_p95 and compute severity_ratio
anomaly_summary = anomaly_summary.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')
anomaly_summary['severity_ratio'] = anomaly_summary['max_value_in_window'] / anomaly_summary['global_p95']

# 6) Produce compact top 20 rows sorted by anomaly_count desc
anomalies_summary_top20 = anomaly_summary.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
anomalies_summary_top20 = anomalies_summary_top20[
    ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'severity_ratio']
]

# 7) For sample anomaly rows: pick top 5 series by anomaly_count, then up to 5 earliest anomaly rows across them
top5_series = anomalies_summary_top20.sort_values('anomaly_count', ascending=False).head(5)[['cmdb_id','kpi_name']]
# merge to flag anomalies in top5
if not top5_series.empty:
    top5_key = set(tuple(x) for x in top5_series.values.tolist())
    mask_top5 = anomalies_df.apply(lambda r: (r['cmdb_id'], r['kpi_name']) in top5_key, axis=1)
    sample_anomalies = anomalies_df[mask_top5].sort_values('timestamp').head(5)[['timestamp','cmdb_id','kpi_name','value']].reset_index(drop=True)
else:
    sample_anomalies = anomalies_df.head(0)[['timestamp','cmdb_id','kpi_name','value']]

# Keep full intermediate results in variables:
# global_p95_df, window_df, window_with_p95, anomalies_df, anomaly_summary

# Final compact outputs
anomalies_summary_top20, sample_anomalies

```
Out[2]:
```
```python
# Summarize the anomaly detection results as a plain-English string.
# Reuse existing kernel variables: anomalies_summary_top20, sample_anomalies

parts = []

# Basic counts
try:
    total_series = int(anomalies_summary_top20.shape[0])
except Exception:
    total_series = 0

parts.append(f"Anomaly detection results for the incident window (2021-03-12 23:30 to 2021-03-13 00:00 UTC+8):")
parts.append(f"Found {total_series} component-kpi series with points >= their global P95 inside the window.")

# Count how many series have global_p95 == 0 (which makes severity undefined)
zero_p95_count = 0
if total_series > 0:
    zero_p95_count = int((anomalies_summary_top20['global_p95'] == 0).sum())
    parts.append(f"{zero_p95_count} series have global_p95 == 0 (severity ratio is undefined or infinite for these).")

# Top series summary (top 5 by anomaly_count)
if total_series == 0:
    parts.append("No anomalies to show.")
else:
    topn = min(5, total_series)
    parts.append(f"Top {topn} series by anomaly_count:")
    for i, row in anomalies_summary_top20.head(topn).iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        cnt = int(row['anomaly_count'])
        gp95 = row['global_p95']
        maxv = row['max_value_in_window']
        sev = row['severity_ratio']
        # format numbers safely
        def fmt(x):
            try:
                if pd.isna(x):
                    return "N/A"
                if float(x) == float('inf'):
                    return "inf"
                return f"{float(x):.6g}"
            except Exception:
                return str(x)
        parts.append(f"- {cmdb} | {kpi} — anomaly_count={cnt}, global_p95={fmt(gp95)}, max_in_window={fmt(maxv)}, severity_ratio={fmt(sev)}")

# Sample anomaly rows (up to 5 earliest across top 5 series)
if 'sample_anomalies' in globals() and not sample_anomalies.empty:
    parts.append("Sample anomaly rows (earliest, up to 5):")
    for _, r in sample_anomalies.iterrows():
        parts.append(f"- ts={int(r['timestamp'])}, {r['cmdb_id']} | {r['kpi_name']} => value={float(r['value']):.6g}")
else:
    parts.append("No sample anomaly rows available.")

summary = " ".join(parts)

# Display the summary string (compact)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    global_p95  anomaly_count  earliest_anomaly_ts  max_value_in_window  severity_ratio
0       IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00             30           1615563000         0.000000e+00             NaN
1       IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00             30           1615563000         0.000000e+00             NaN
2       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.796410e+07             30           1615563000         5.796410e+07             1.0
3   apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00             30           1615563000         7.000000e+00             1.0
4    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01             30           1615563000         4.000000e-01             1.0
5    Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00             30           1615563000         7.000000e+00             1.0
6    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00             30           1615563000         0.000000e+00             NaN
7    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00             30           1615563000         0.000000e+00             NaN
8   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  0.000000e+00             30           1615563000         0.000000e+00             NaN
9   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  0.000000e+00             30           1615563000         0.000000e+00             NaN
10   Redis01             redis-Redis_6379_Redis  (expired_keys)  6.000000e+00             30           1615563000         6.000000e+00             1.0
11  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00             30           1615563000         0.000000e+00             NaN
12  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  4.000000e+00             30           1615563000         4.000000e+00             1.0
13  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02             30           1615563000         1.000000e+02             1.0
14  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter  0.000000e+00             30           1615563000         0.000000e+00             NaN
15  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02             30           1615563000         5.000000e+02             1.0
16  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02             30           1615563000         1.000000e+02             1.0
17   Redis01             redis-Redis_6379_Redis  (evicted_keys)  0.000000e+00             30           1615563000         0.000000e+00             NaN
18      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09             30           1615563000         2.040136e+09             1.0
19   Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00             30           1615563000         1.000000e+00             inf,     timestamp   cmdb_id                                           kpi_name       value
0  1615563000      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  57964104.0
1  1615563000      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         0.0
2  1615563000      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         0.0
3  1615563000   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...         0.4
4  1615563000  apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         7.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# ensure types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# allowed components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# 1) Compute global P95 per (cmdb_id, trace_name) using the full trace_df (global threshold BEFORE window filtering)
global_p95_trace_df = trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95)
global_p95_trace_df = global_p95_trace_df.rename(columns={'value': 'global_p95'})

# restrict to allowed components
global_p95_trace_df = global_p95_trace_df[global_p95_trace_df['cmdb_id'].isin(allowed_components)].reset_index(drop=True)

# 2) Define incident window in UTC+8 (Asia/Shanghai) and get epoch seconds
start_ts = int(pd.Timestamp('2021-03-12 23:30:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-13 00:00:00', tz=tz).timestamp())

# 3) Extract windowed trace data for allowed components
window_trace_df = trace_df[
    (trace_df['cmdb_id'].isin(allowed_components)) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# 4) Merge global_p95 into windowed rows
window_with_p95_trace = window_trace_df.merge(global_p95_trace_df, on=['cmdb_id', 'trace_name'], how='left')

# 5) Identify anomaly points where value >= global_p95 (exclude rows with missing global_p95)
anomalies_trace_df = window_with_p95_trace[
    window_with_p95_trace['global_p95'].notna() &
    (window_with_p95_trace['value'] >= window_with_p95_trace['global_p95'])
].copy()

# 6) Aggregate anomalies per (cmdb_id, trace_name)
anomaly_trace_summary = anomalies_trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    anomaly_count=('timestamp', 'size'),
    earliest_anomaly_ts=('timestamp', 'min'),
    max_value_in_window=('value', 'max')
)

# attach global_p95
anomaly_trace_summary = anomaly_trace_summary.merge(global_p95_trace_df, on=['cmdb_id', 'trace_name'], how='left')

# compute severity_ratio with handling for global_p95 == 0
# If global_p95 == 0:
#   - if max_value_in_window == 0 -> severity_ratio = NaN (undefined)
#   - if max_value_in_window > 0 -> severity_ratio = inf
anomaly_trace_summary['severity_ratio'] = np.where(
    anomaly_trace_summary['global_p95'] == 0,
    np.where(anomaly_trace_summary['max_value_in_window'] == 0, np.nan, np.inf),
    anomaly_trace_summary['max_value_in_window'] / anomaly_trace_summary['global_p95']
)

# 7) Compact top 20 rows sorted by anomaly_count desc
anomaly_trace_summary_top20 = anomaly_trace_summary.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
anomaly_trace_summary_top20 = anomaly_trace_summary_top20[
    ['cmdb_id', 'trace_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'severity_ratio']
]

# 8) Sample anomaly rows: choose top 5 trace series by anomaly_count, then up to 5 earliest anomaly rows across them
top5_trace_series = anomaly_trace_summary.sort_values('anomaly_count', ascending=False).head(5)[['cmdb_id','trace_name']]
if not top5_trace_series.empty:
    top5_keys = set(tuple(x) for x in top5_trace_series.values.tolist())
    mask_top5 = anomalies_trace_df.apply(lambda r: (r['cmdb_id'], r['trace_name']) in top5_keys, axis=1)
    sample_trace_anomalies = anomalies_trace_df[mask_top5].sort_values('timestamp').head(5)[['timestamp','cmdb_id','trace_name','value']].reset_index(drop=True)
else:
    sample_trace_anomalies = anomalies_trace_df.head(0)[['timestamp','cmdb_id','trace_name','value']]

# Keep full intermediate results in memory (variables):
# global_p95_trace_df, window_trace_df, window_with_p95_trace, anomalies_trace_df, anomaly_trace_summary

# Final compact outputs
anomaly_trace_summary_top20, sample_trace_anomalies

```
Out[3]:
```
```python
# Summarize trace anomaly detection results (reuse kernel variables)
# Variables from kernel: anomaly_trace_summary_top20, sample_trace_anomalies

import pandas as pd
import math

parts = []
parts.append("Trace anomaly detection summary for incident window (2021-03-12 23:30 to 2021-03-13 00:00 UTC+8):")

# basic counts
total_series = int(anomaly_trace_summary_top20.shape[0]) if 'anomaly_trace_summary_top20' in globals() else 0
parts.append(f"Found {total_series} trace series with values >= their global P95 in the window.")

# count zero global_p95
zero_p95_count = 0
if total_series > 0:
    zero_p95_count = int((anomaly_trace_summary_top20['global_p95'] == 0).sum())
    if zero_p95_count:
        parts.append(f"{zero_p95_count} series have global_p95 == 0 (severity ratio undefined or infinite for these).")

# top 5 series by anomaly_count
if total_series > 0:
    topn = min(5, total_series)
    parts.append(f"Top {topn} trace series (cmdb_id | trace_name — anomaly_count, global_p95, max_in_window, severity_ratio):")
    for _, r in anomaly_trace_summary_top20.head(topn).iterrows():
        def fmt(x):
            if pd.isna(x):
                return "N/A"
            if isinstance(x, (float, int)):
                if math.isinf(x):
                    return "inf"
                return f"{float(x):.6g}"
            return str(x)
        parts.append(f"- {r['cmdb_id']} | {r['trace_name']} — {int(r['anomaly_count'])}, gp95={fmt(r['global_p95'])}, max={fmt(r['max_value_in_window'])}, sev={fmt(r['severity_ratio'])}")

# sample anomaly rows (up to 5 earliest across top5)
if 'sample_trace_anomalies' in globals() and not sample_trace_anomalies.empty:
    parts.append("Sample anomaly rows (earliest, up to 5):")
    for _, s in sample_trace_anomalies.iterrows():
        parts.append(f"- ts={int(s['timestamp'])}, {s['cmdb_id']} | {s['trace_name']} => value={float(s['value']):.6g}")
else:
    parts.append("No sample anomaly rows available.")

summary = " ".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name  global_p95  anomaly_count  earliest_anomaly_ts  max_value_in_window  severity_ratio
0     MG01    trace.from_dockerA1.row_count   10.000000              6           1615563420            14.000000        1.400000
1     MG01    trace.from_dockerA2.row_count   14.000000              4           1615564200            16.000000        1.142857
2     MG02    trace.from_dockerA1.row_count   14.000000              3           1615563480            16.000000        1.142857
3     MG02      trace.to_dockerB1.row_count   52.000000              3           1615564140            86.000000        1.653846
4     MG01      trace.to_dockerA2.row_count   41.000000              3           1615563540            50.000000        1.219512
5     IG01      trace.to_Tomcat01.row_count   59.200000              2           1615564680            65.000000        1.097973
6     IG01  trace.to_Tomcat02.duration_mean    0.176123              2           1615564080             0.223068        1.266548
7     IG01   trace.to_Tomcat01.duration_p95    1.174050              2           1615563480             1.581400        1.346961
8     IG01             trace.self.row_count  406.100000              2           1615564680           596.000000        1.467619
9     IG01  trace.to_Tomcat01.duration_mean    0.254133              2           1615563480             0.270600        1.064795
10    IG02         trace.self.duration_mean    0.146907              2           1615563720             0.343475        2.338049
11    IG02  trace.to_Tomcat01.duration_mean    0.149307              2           1615563720             0.266826        1.787098
12    IG02             trace.self.row_count  388.300000              2           1615564680           606.000000        1.560649
13    IG01   trace.to_Tomcat03.duration_p95    0.895770              2           1615564500             1.022600        1.141588
14    IG01      trace.to_Tomcat03.row_count   49.550000              2           1615564560            97.000000        1.957619
15    IG01  trace.to_Tomcat04.duration_mean    0.202408              2           1615563720             0.224415        1.108726
16    IG01   trace.to_Tomcat04.duration_p95    1.090625              2           1615563720             1.133750        1.039542
17    IG01      trace.to_Tomcat04.row_count   49.550000              2           1615564560            54.000000        1.089808
18    IG02   trace.to_Tomcat01.duration_p95    0.783705              2           1615564500             0.853000        1.088420
19    IG02          trace.self.duration_p95    0.987752              2           1615564500             1.954000        1.978228,     timestamp cmdb_id                     trace_name  value
0  1615563420    MG01  trace.from_dockerA1.row_count   10.0
1  1615563480    MG02  trace.from_dockerA1.row_count   14.0
2  1615563540    MG01    trace.to_dockerA2.row_count   41.0
3  1615563600    MG02  trace.from_dockerA1.row_count   14.0
4  1615563780    MG01  trace.from_dockerA1.row_count   10.0)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Components to analyze (as requested)
components = ['IG01','IG02','MG01','MG02','Tomcat02','Tomcat03','Tomcat04',
              'apache01','apache02','Mysql01','Mysql02','Redis01']

# 1) Compute global P95 per (cmdb_id, kpi_name) using full metric_df (global BEFORE window filtering)
global_p95_metrics = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95)
global_p95_metrics = global_p95_metrics.rename(columns={'value':'global_p95'})

# Restrict global thresholds to the selected components (but keep full table too)
global_p95_sel = global_p95_metrics[global_p95_metrics['cmdb_id'].isin(components)].reset_index(drop=True)

# 2) Define incident window (UTC+8)
start_ts = int(pd.Timestamp('2021-03-12 23:30:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-13 00:00:00', tz=tz).timestamp())

# 3) Extract window data for selected components
window_df = metric_df[
    (metric_df['cmdb_id'].isin(components)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Merge global p95 into window rows
window_with_p95 = window_df.merge(global_p95_sel, on=['cmdb_id','kpi_name'], how='left')

# 4) Determine anomalous points:
# - If global_p95 is NaN -> cannot evaluate -> treat as non-anomalous
# - If global_p95 == 0 -> anomaly if value != 0 (detect consecutive non-zero runs), but severity_ratio will be NaN
# - Else -> anomaly if value >= global_p95
def is_anomaly(row):
    gp = row['global_p95']
    v = row['value']
    if pd.isna(gp) or pd.isna(v):
        return False
    if gp == 0:
        return v != 0
    return v >= gp

window_with_p95['anomaly'] = window_with_p95.apply(is_anomaly, axis=1)

# 5) Identify consecutive anomalous sub-series (min length 2). Consecutive means timestamp difference == 60 seconds.
# We'll build segments dataframe for all series
segments = []
grouped = window_with_p95.groupby(['cmdb_id','kpi_name'], as_index=False)

for (cmdb, kpi), grp in grouped:
    g = grp[grp['anomaly']].sort_values('timestamp')
    if g.empty:
        continue
    # compute time diffs
    g = g.reset_index(drop=True)
    g['diff'] = g['timestamp'].diff().fillna(9999)
    # new segment when diff != 60
    g['seg_id'] = (g['diff'] != 60).cumsum()
    for seg_id, seg in g.groupby('seg_id'):
        length = int(seg.shape[0])
        if length >= 2:
            start_ts_seg = int(seg['timestamp'].iloc[0])
            end_ts_seg = int(seg['timestamp'].iloc[-1])
            max_val = float(seg['value'].max())
            # find global_p95 for this series (may be NaN if missing)
            gp_row = global_p95_sel[
                (global_p95_sel['cmdb_id']==cmdb) & (global_p95_sel['kpi_name']==kpi)
            ]
            gp95 = float(gp_row['global_p95'].iloc[0]) if not gp_row.empty else np.nan
            # severity ratio handling: if gp95 == 0 => severity NaN (ignored), else ratio = max_val / gp95
            if pd.isna(gp95) or gp95 == 0:
                severity = np.nan
            else:
                severity = max_val / gp95 if gp95 != 0 else np.nan
            segments.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'segment_start_ts': start_ts_seg,
                'segment_end_ts': end_ts_seg,
                'length': length,
                'max_value': max_val,
                'global_p95': gp95,
                'severity_ratio': severity
            })

# Full segments table (intermediate)
segments_df = pd.DataFrame(segments)

# 6) Aggregate per (cmdb_id, kpi_name)
if not segments_df.empty:
    # segments_count
    agg = segments_df.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        segments_count=('length','size'),
        earliest_segment_start_ts=('segment_start_ts','min')
    )
    # For earliest segment details, merge to get end and length corresponding to earliest start
    earliest = segments_df.sort_values(['segment_start_ts']).groupby(['cmdb_id','kpi_name'], as_index=False).first()
    earliest = earliest.rename(columns={
        'segment_end_ts':'earliest_segment_end_ts',
        'length':'earliest_segment_length'
    })[['cmdb_id','kpi_name','earliest_segment_end_ts','earliest_segment_length']]
    agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    # max_segment_severity_ratio (ignore NaNs)
    max_sev = segments_df.groupby(['cmdb_id','kpi_name'], as_index=False)['severity_ratio'].agg(lambda x: np.nan if x.dropna().empty else float(x.dropna().max()))
    max_sev = max_sev.rename(columns={'severity_ratio':'max_segment_severity_ratio'})
    agg = agg.merge(max_sev, on=['cmdb_id','kpi_name'], how='left')

    # Attach global_p95 for clarity (from global_p95_sel)
    agg = agg.merge(global_p95_sel, on=['cmdb_id','kpi_name'], how='left')
    # reorder columns
    agg = agg[['cmdb_id','kpi_name','global_p95','segments_count','earliest_segment_start_ts','earliest_segment_end_ts','earliest_segment_length','max_segment_severity_ratio']]
else:
    agg = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','segments_count','earliest_segment_start_ts','earliest_segment_end_ts','earliest_segment_length','max_segment_severity_ratio'])

# 7) Compact results: top 20 rows sorted by segments_count desc
agg_top20 = agg.sort_values('segments_count', ascending=False).head(20).reset_index(drop=True)

# 8) Sample segments: up to 10 sample segments (one row per segment). Choose longest segments first, then earliest.
if not segments_df.empty:
    sample_segments = segments_df.sort_values(['length','segment_start_ts'], ascending=[False, True]).head(10).reset_index(drop=True)
    sample_segments = sample_segments[['cmdb_id','kpi_name','segment_start_ts','segment_end_ts','length','max_value','severity_ratio','global_p95']]
else:
    sample_segments = segments_df.copy()

# Keep full intermediates: global_p95_metrics, global_p95_sel, window_with_p95, segments_df

# Final compact outputs
agg_top20, sample_segments

```
Out[4]:
```
```python
# Summarize the consecutive-anomaly segment analysis for the requested components/window.
# Reuse kernel variables: agg_top20, sample_segments, segments_df

import pandas as pd
import math

# Build summary parts
parts = []
parts.append("Metric consecutive-anomaly segment analysis (2021-03-12 23:30 to 2021-03-13 00:00 UTC+8):")

# overall counts
total_series_with_segments = int(agg_top20.shape[0]) if 'agg_top20' in globals() else 0
total_segments = int(segments_df.shape[0]) if 'segments_df' in globals() else 0
parts.append(f"Found {total_series_with_segments} (cmdb_id, kpi_name) with >=1 anomalous consecutive segment (min length 2). Total detected segments: {total_segments}.")

# pattern summary
parts.append("Most detected segments are short (length=2 points) and many series have multiple small segments (segments_count up to 10).")
parts.append("Severity ratios (max_value / global_p95) are generally 1.0 for top series, indicating values reached but did not substantially exceed the global P95 in these segments.")

# Top examples (from agg_top20)
if total_series_with_segments > 0:
    parts.append("Top example series (cmdb_id | kpi_name — segments_count, earliest segment start->end, max_segment_severity_ratio):")
    for _, r in agg_top20.head(5).iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        sc = int(r['segments_count'])
        sst = int(r['earliest_segment_start_ts']) if not pd.isna(r['earliest_segment_start_ts']) else "N/A"
        sed = int(r['earliest_segment_end_ts']) if not pd.isna(r['earliest_segment_end_ts']) else "N/A"
        sev = r['max_segment_severity_ratio']
        sev_fmt = ("N/A" if pd.isna(sev) else ("inf" if math.isinf(sev) else f"{float(sev):.3g}"))
        parts.append(f"- {cmdb} | {kpi} — {sc} segments, earliest {sst}->{sed}, max_severity={sev_fmt}")

# Sample segments (up to 10)
if 'sample_segments' in globals() and not sample_segments.empty:
    parts.append("Sample segments (up to 10, one row per segment):")
    for _, s in sample_segments.head(10).iterrows():
        cmdb = s['cmdb_id']
        kpi = s['kpi_name']
        ss = int(s['segment_start_ts'])
        se = int(s['segment_end_ts'])
        ln = int(s['length'])
        mv = float(s['max_value'])
        gp = s.get('global_p95', float('nan'))
        sev = s['severity_ratio']
        if pd.isna(sev):
            sev_fmt = "N/A"
        else:
            sev_fmt = ("inf" if math.isinf(sev) else f"{float(sev):.3g}")
        parts.append(f"- {cmdb} | {kpi} : {ss}->{se}, len={ln}, max={mv:.6g}, gp95={gp if not pd.isna(gp) else 'N/A'}, sev={sev_fmt}")
else:
    parts.append("No sample segments available.")

# Notable long runs
# detect any segments that span the full window (length 30) in segments_df
long_runs = []
if 'segments_df' in globals() and not segments_df.empty:
    long_runs = segments_df[segments_df['length'] >= 30]
    if not long_runs.empty:
        parts.append("Notably, some JVM memory/process series show continuous anomaly across the whole window (length=30). These are full-window reaches of global P95 with severity ~1.0.")

summary = " ".join(parts)

# Display the concise plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                       kpi_name  global_p95  segments_count  earliest_segment_start_ts  earliest_segment_end_ts  earliest_segment_length  max_segment_severity_ratio
0       IG01   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        98.0              10                 1615563060               1615563120                        2                         1.0
1       MG01     OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         3.0              10                 1615563000               1615563060                        2                         1.0
2       MG01   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        97.0              10                 1615563000               1615563060                        2                         1.0
3       MG02   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        97.0              10                 1615563000               1615563060                        2                         1.0
4       MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem        58.0              10                 1615563000               1615563060                        2                         1.0
5       MG01  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         2.0              10                 1615563000               1615563060                        2                         1.0
6    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem        54.0              10                 1615563060               1615563120                        2                         1.0
7    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem        54.0              10                 1615563000               1615563060                        2                         1.0
8   Tomcat02   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        98.0              10                 1615563000               1615563060                        2                         1.0
9   apache01   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        68.0              10                 1615563060               1615563120                        2                         1.0
10  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem       124.0              10                 1615563060               1615563120                        2                         1.0
11  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem        58.0               9                 1615563060               1615563120                        2                         1.0
12  Tomcat03     OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         3.0               9                 1615563060               1615563120                        2                         1.0
13  Tomcat04     OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         3.0               9                 1615563060               1615563120                        2                         1.0
14  Tomcat03  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         1.0               9                 1615563060               1615563120                        2                         1.0
15  Tomcat04   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        98.0               9                 1615563000               1615563060                        2                         1.0
16  Tomcat04  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         1.0               9                 1615563060               1615563120                        2                         1.0
17   Mysql02     Mysql-MySQL_3306_GetConnectedStateOfMysqld         1.0               9                 1615563120               1615563180                        2                         1.0
18  apache02   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        74.0               9                 1615563000               1615563060                        2                         1.0
19  Tomcat03      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem        58.0               9                 1615563060               1615563120                        2                         1.0,    cmdb_id                                           kpi_name  segment_start_ts  segment_end_ts  length     max_value  severity_ratio    global_p95
0     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax        1615563000      1615564740      30  2.040136e+09             1.0  2.040136e+09
1     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed        1615563000      1615564740      30  5.796410e+07             1.0  5.796410e+07
2     IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax        1615563000      1615564740      30  2.040136e+09             1.0  2.040136e+09
3     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax        1615563000      1615564740      30  2.040136e+09             1.0  2.040136e+09
4     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed        1615563000      1615564740      30  5.069495e+08             1.0  5.069495e+08
5     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax        1615563000      1615564740      30  2.040136e+09             1.0  2.040136e+09
6     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed        1615563000      1615564740      30  9.471521e+08             1.0  9.471521e+08
7  Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...        1615563000      1615564740      30  4.000000e-01             1.0  4.000000e-01
8  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount        1615563000      1615564740      30  7.000000e+00             1.0  7.000000e+00
9  Redis01        redis-Redis_6379_Redis  (connected_clients)        1615563000      1615564740      30  6.020000e+02             1.0  6.020000e+02)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing data/variables from kernel: trace_df, global_p95_trace_df, tz if available
# If tz not present, set it
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Target series list
targets = [
    ('IG02', 'trace.self.duration_mean'),
    ('IG02', 'trace.self.duration_p95'),
    ('IG02', 'trace.to_Tomcat01.duration_mean'),
    ('IG01', 'trace.to_Tomcat01.duration_p95'),
    ('IG01', 'trace.to_Tomcat01.duration_mean'),
]

targets_df = pd.DataFrame(targets, columns=['cmdb_id','trace_name'])

# Ensure global_p95_trace_df exists and contains global P95 computed on full trace_df
try:
    gp95_all = global_p95_trace_df.copy()
except NameError:
    gp95_all = trace_df.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# For requested targets, get global_p95; if missing compute from full trace_df
gp95_targets = targets_df.merge(gp95_all, on=['cmdb_id','trace_name'], how='left')

# Find missing and compute
missing_mask = gp95_targets['global_p95'].isna()
if missing_mask.any():
    missing_pairs = gp95_targets[missing_mask][['cmdb_id','trace_name']]
    if not missing_pairs.empty:
        comp = trace_df.merge(missing_pairs, on=['cmdb_id','trace_name'], how='inner')
        if not comp.empty:
            comp_gp95 = comp.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})
            gp95_targets = gp95_targets.drop(columns=['global_p95']).merge(comp_gp95, on=['cmdb_id','trace_name'], how='left')

# Define incident window in UTC+8
start_ts = int(pd.Timestamp('2021-03-12 23:30:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-13 00:00:00', tz=tz).timestamp())

# Filter trace_df for targets within window
window_rows = trace_df[
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].merge(targets_df, on=['cmdb_id','trace_name'], how='inner').copy()

# Attach global_p95 to each row
window_with_gp95 = window_rows.merge(gp95_targets, on=['cmdb_id','trace_name'], how='left')

# Limit returned rows to at most 100 (return all if <=100)
if window_with_gp95.shape[0] > 100:
    selected_rows_df = window_with_gp95[['cmdb_id','trace_name','timestamp','value','global_p95']].sort_values(['cmdb_id','trace_name','timestamp']).head(100).reset_index(drop=True)
else:
    selected_rows_df = window_with_gp95[['cmdb_id','trace_name','timestamp','value','global_p95']].sort_values(['cmdb_id','trace_name','timestamp']).reset_index(drop=True)

# Per-series summary: earliest timestamp in window where value >= global_p95, max value in window, severity ratio handling gp95==0
def compute_series_summary(df_rows, gp95_df):
    out_rows = []
    grouped = df_rows.groupby(['cmdb_id','trace_name'], as_index=False)
    for _, g in grouped:
        cmdb = g['cmdb_id'].iloc[0]
        trace_name = g['trace_name'].iloc[0]
        gp95_row = gp95_df[(gp95_df['cmdb_id']==cmdb) & (gp95_df['trace_name']==trace_name)]
        gp95 = float(gp95_row['global_p95'].iloc[0]) if not gp95_row.empty and not pd.isna(gp95_row['global_p95'].iloc[0]) else np.nan
        max_val = float(g['value'].max()) if not g['value'].isna().all() else np.nan
        # earliest ts where value >= gp95 (if gp95 is NaN -> None)
        if pd.isna(gp95):
            earliest_ts = np.nan
        else:
            if gp95 == 0:
                # treat anomaly as value != 0 per prior convention
                cond = g['value'] != 0
            else:
                cond = g['value'] >= gp95
            matched = g[cond]
            earliest_ts = int(matched['timestamp'].min()) if not matched.empty else np.nan
        # severity ratio handling
        if pd.isna(gp95):
            severity = np.nan
        else:
            if gp95 == 0:
                if pd.isna(max_val) or max_val == 0:
                    severity = np.nan
                else:
                    severity = np.inf
            else:
                severity = max_val / gp95 if not pd.isna(max_val) else np.nan
        out_rows.append({
            'cmdb_id': cmdb,
            'trace_name': trace_name,
            'global_p95': gp95,
            'earliest_anomaly_ts_in_window': earliest_ts,
            'max_value_in_window': max_val,
            'severity_ratio': severity
        })
    return pd.DataFrame(out_rows)

# Build summary for all requested series (ensure series with no rows in window are also included with NaNs)
# start from gp95_targets to include all target series
summary_df = gp95_targets.merge(
    window_with_gp95.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        max_value_in_window=('value','max')
    ),
    on=['cmdb_id','trace_name'],
    how='left'
)

# Compute earliest anomaly ts per series
earliest_list = []
for _, row in gp95_targets.iterrows():
    cmdb = row['cmdb_id']
    tname = row['trace_name']
    gp95 = row['global_p95']
    subset = window_with_gp95[(window_with_gp95['cmdb_id']==cmdb) & (window_with_gp95['trace_name']==tname)]
    if subset.empty:
        earliest = np.nan
        maxval = np.nan
    else:
        maxval = float(subset['value'].max())
        if pd.isna(gp95):
            earliest = np.nan
        else:
            if gp95 == 0:
                cond = subset['value'] != 0
            else:
                cond = subset['value'] >= gp95
            matched = subset[cond]
            earliest = int(matched['timestamp'].min()) if not matched.empty else np.nan
    # severity ratio
    if pd.isna(gp95):
        sev = np.nan
    else:
        if gp95 == 0:
            if pd.isna(maxval) or maxval == 0:
                sev = np.nan
            else:
                sev = np.inf
        else:
            sev = maxval / gp95 if not pd.isna(maxval) else np.nan
    earliest_list.append({'cmdb_id': cmdb, 'trace_name': tname, 'global_p95': gp95,
                          'earliest_anomaly_ts_in_window': earliest,
                          'max_value_in_window': maxval,
                          'severity_ratio': sev})

series_summary_df = pd.DataFrame(earliest_list)[['cmdb_id','trace_name','global_p95','earliest_anomaly_ts_in_window','max_value_in_window','severity_ratio']]

# Final compact outputs: selected_rows_df (<=100 rows) and series_summary_df (one row per target series)
selected_rows_df, series_summary_df

```
Out[5]:
```
```python
# Build a concise plain-English summary using existing kernel results:
# selected_rows_df, series_summary_df

import math
import pandas as pd
import numpy as np

# Use series_summary_df from the kernel (one row per requested series)
df = series_summary_df.copy()

parts = []
parts.append("Trace series within incident window (2021-03-12 23:30 to 2021-03-13 00:00 UTC+8):")

# Note about returned rows
count_rows = int(selected_rows_df.shape[0]) if 'selected_rows_df' in globals() else 0
parts.append(f"Retrieved up to {count_rows} per-minute rows (capped at 100 for display).")

# Per-series findings
parts.append("Per-series summary (global_p95, earliest timestamp in window where value >= global_p95, max value in window, severity_ratio):")
for _, r in df.iterrows():
    cmdb = r['cmdb_id']
    name = r['trace_name']
    gp95 = r['global_p95']
    earliest = r['earliest_anomaly_ts_in_window']
    maxv = r['max_value_in_window']
    sev = r['severity_ratio']
    # formatting
    def fmtnum(x):
        if pd.isna(x):
            return "N/A"
        if isinstance(x, (float, int)):
            if math.isinf(x):
                return "inf"
            return f"{float(x):.6g}"
        return str(x)
    parts.append(f"- {cmdb} | {name} — gp95={fmtnum(gp95)}, earliest_anomaly_ts={int(earliest) if not pd.isna(earliest) else 'N/A'}, max_value={fmtnum(maxv)}, severity_ratio={fmtnum(sev)}")

# Short interpretation
parts.append("Interpretation: All five requested trace series showed values exceeding their global P95 inside the window.")
parts.append("Notable strong deviations: IG02.trace.self.duration_mean (severity ~2.34), IG02.trace.self.duration_p95 (severity ~1.98), and IG02.trace.to_Tomcat01.duration_mean (severity ~1.79). IG01.to_Tomcat01 series also exceeded P95 (severity ~1.35 and ~1.06).")

summary = " ".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name   timestamp     value  global_p95
0     IG01  trace.to_Tomcat01.duration_mean  1615563000  0.076677    0.254133
1     IG01  trace.to_Tomcat01.duration_mean  1615563060  0.098138    0.254133
2     IG01  trace.to_Tomcat01.duration_mean  1615563120  0.083231    0.254133
3     IG01  trace.to_Tomcat01.duration_mean  1615563180  0.049367    0.254133
4     IG01  trace.to_Tomcat01.duration_mean  1615563240  0.053318    0.254133
..     ...                              ...         ...       ...         ...
95    IG02          trace.self.duration_p95  1615563300  0.158300    0.987752
96    IG02          trace.self.duration_p95  1615563360  0.285700    0.987752
97    IG02          trace.self.duration_p95  1615563420  0.373900    0.987752
98    IG02          trace.self.duration_p95  1615563480  0.553000    0.987752
99    IG02          trace.self.duration_p95  1615563540  0.539450    0.987752

[100 rows x 5 columns],   cmdb_id                       trace_name  global_p95  earliest_anomaly_ts_in_window  max_value_in_window  severity_ratio
0    IG02         trace.self.duration_mean    0.146907                     1615563720             0.343475        2.338049
1    IG02          trace.self.duration_p95    0.987752                     1615564500             1.954000        1.978228
2    IG02  trace.to_Tomcat01.duration_mean    0.149307                     1615563720             0.266826        1.787098
3    IG01   trace.to_Tomcat01.duration_p95    1.174050                     1615563480             1.581400        1.346961
4    IG01  trace.to_Tomcat01.duration_mean    0.254133                     1615563480             0.270600        1.064795)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure correct dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 for IG02 using full series
ig02_full = metric_df[metric_df['cmdb_id'] == 'IG02'].copy()
global_p95_ig02 = ig02_full.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# 2) Incident window in UTC+8
start_ts = int(pd.Timestamp('2021-03-12 23:30:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-13 00:00:00', tz=tz).timestamp())

# 3) Filter IG02 points inside the window
window_ig02 = ig02_full[(ig02_full['timestamp'] >= start_ts) & (ig02_full['timestamp'] <= end_ts)].copy()

# 4) Merge global p95 into windowed rows
window_with_p95 = window_ig02.merge(global_p95_ig02, on=['cmdb_id','kpi_name'], how='left')

# 5) Detect anomalies per prior rule:
#    - If global_p95 is NaN => cannot detect (False)
#    - If global_p95 == 0 => anomaly if value != 0
#    - Else anomaly if value >= global_p95
def detect_anom(row):
    gp = row['global_p95']
    v = row['value']
    if pd.isna(gp) or pd.isna(v):
        return False
    if gp == 0:
        return v != 0
    return v >= gp

window_with_p95['is_anomaly'] = window_with_p95.apply(detect_anom, axis=1)

# 6) Aggregate per (cmdb_id, kpi_name) manually (to include only series present in global_p95_ig02)
groups = []
# collect series that have window rows
for (cmdb, kpi), g in window_with_p95.groupby(['cmdb_id','kpi_name']):
    an_count = int(g['is_anomaly'].sum())
    earliest = np.nan
    if an_count > 0:
        earliest = int(g.loc[g['is_anomaly'], 'timestamp'].min())
    maxv = float(g['value'].max()) if not g['value'].isna().all() else np.nan
    gp95 = float(global_p95_ig02[(global_p95_ig02['cmdb_id']==cmdb) & (global_p95_ig02['kpi_name']==kpi)]['global_p95'].iloc[0]) \
           if not global_p95_ig02[(global_p95_ig02['cmdb_id']==cmdb) & (global_p95_ig02['kpi_name']==kpi)].empty else np.nan
    groups.append({'cmdb_id':cmdb,'kpi_name':kpi,'global_p95':gp95,
                   'anomaly_count_in_window':an_count,
                   'earliest_anomaly_ts_in_window':earliest,
                   'max_value_in_window':maxv})

agg_anom = pd.DataFrame(groups)

# 7) Ensure all IG02 series are present in summary (series with no points in window)
all_series = global_p95_ig02[['cmdb_id','kpi_name','global_p95']].copy()
summary = all_series.merge(agg_anom, on=['cmdb_id','kpi_name','global_p95'], how='left')

# Fill missing counts with 0, keep earliest and max as NaN where absent
summary['anomaly_count_in_window'] = summary['anomaly_count_in_window'].fillna(0).astype(int)
summary['earliest_anomaly_ts_in_window'] = summary['earliest_anomaly_ts_in_window'].where(~summary['earliest_anomaly_ts_in_window'].isna(), np.nan)
summary['max_value_in_window'] = summary['max_value_in_window'].astype(float)

# 8) Compute severity_ratio with handling for global_p95 == 0 and NaNs
def compute_sev(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp):
        return np.nan
    if gp == 0:
        if pd.isna(mv) or mv == 0:
            return np.nan
        return np.inf
    if pd.isna(mv):
        return np.nan
    return mv / gp

summary['severity_ratio'] = summary.apply(compute_sev, axis=1)

# 9) Sort and return top 20 by anomaly_count desc then severity_ratio desc
summary_top20 = summary.sort_values(['anomaly_count_in_window','severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# 10) Sample anomaly rows: top 5 series by (anomaly_count, severity_ratio), then earliest anomaly rows across them (up to 5)
top5 = summary_top20.sort_values(['anomaly_count_in_window','severity_ratio'], ascending=[False, False]).head(5)[['cmdb_id','kpi_name']]
top5_keys = set([tuple(x) for x in top5.values.tolist()])

anomalies_df = window_with_p95[window_with_p95['is_anomaly']].copy()
# filter to top5 series
anomalies_top5 = anomalies_df[anomalies_df.apply(lambda r: (r['cmdb_id'], r['kpi_name']) in top5_keys, axis=1)]
sample_anomalies = anomalies_top5.sort_values('timestamp')[['timestamp','cmdb_id','kpi_name','value']].head(5).reset_index(drop=True)

# Keep intermediate variables in kernel:
# global_p95_ig02, window_ig02, window_with_p95, anomalies_df, agg_anom, summary

# Final compact outputs
summary_top20, sample_anomalies

```
Out[6]:
```
```python
# Summarize IG02 metric anomaly analysis results (plain English)
import math
import pandas as pd
import numpy as np

# Reuse kernel variables: summary_top20, sample_anomalies
try:
    df = summary_top20.copy()
    samples = sample_anomalies.copy()
except NameError:
    raise RuntimeError("Expected kernel variables 'summary_top20' and 'sample_anomalies' not found.")

parts = []
parts.append("IG02 metric anomaly summary for incident window (2021-03-12 23:30 to 2021-03-13 00:00 UTC+8):")

total_series = int(df.shape[0])
parts.append(f"- Total IG02 KPI series inspected (top listed): {total_series}.")

# How many series had at least one anomaly in the window
with_anom = int((df['anomaly_count_in_window'] > 0).sum())
parts.append(f"- Series with >=1 point >= global P95 in window: {with_anom}.")

# Top series by anomaly_count
if with_anom > 0:
    max_count = int(df['anomaly_count_in_window'].max())
    top_by_count = df[df['anomaly_count_in_window'] == max_count].head(5)
    top_list = []
    for _, r in top_by_count.iterrows():
        top_list.append(f"{r['kpi_name']} (count={int(r['anomaly_count_in_window'])})")
    parts.append(f"- Top anomaly count = {max_count}. Example series: " + "; ".join(top_list) + ".")

# Severity overview
# treat NaN and inf appropriately
def fmt(x):
    if pd.isna(x):
        return "N/A"
    if math.isinf(x):
        return "inf"
    return f"{float(x):.3g}"

sev_gt1 = int((df['severity_ratio'] > 1).sum())
parts.append(f"- Series with severity_ratio > 1: {sev_gt1} (most series show severity_ratio = 1.0, meaning values reached P95 but did not greatly exceed it).")

# Earliest anomaly timestamp commonality (many show earliest at start of window)
earliest_ts_mode = int(df['earliest_anomaly_ts_in_window'].dropna().mode().iloc[0]) if not df['earliest_anomaly_ts_in_window'].dropna().empty else None
if earliest_ts_mode:
    parts.append(f"- Many series first exceeded global P95 at or near timestamp {earliest_ts_mode} (epoch seconds).")

# Provide a few concrete examples (top 5 rows)
parts.append("- Example top series rows (cmdb_id | kpi_name — gp95, anomaly_count, earliest_ts, max_value, severity):")
for _, r in df.head(5).iterrows():
    parts.append(f"  - IG02 | {r['kpi_name']} — gp95={fmt(r['global_p95'])}, count={int(r['anomaly_count_in_window'])}, earliest={int(r['earliest_anomaly_ts_in_window']) if not pd.isna(r['earliest_anomaly_ts_in_window']) else 'N/A'}, max={fmt(r['max_value_in_window'])}, sev={fmt(r['severity_ratio'])}")

# Sample anomaly rows (up to 5 earliest across top 5 series)
if not samples.empty:
    parts.append("Sample anomaly rows (earliest, up to 5):")
    for _, s in samples.iterrows():
        parts.append(f"  - ts={int(s['timestamp'])}, {s['cmdb_id']} | {s['kpi_name']} => value={float(s['value']):.6g}")
else:
    parts.append("No sample anomaly rows available.")

# Short interpretation
parts.append("Interpretation: IG02 shows many KPIs reaching their global P95 inside the window (several series with counts near full-window), but severity ratios are mostly ~1.0, indicating values met the P95 threshold rather than spiking far beyond it.")

summary = " ".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name    global_p95  anomaly_count_in_window  earliest_anomaly_ts_in_window  max_value_in_window  severity_ratio
0     IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                       30                   1.615563e+09         2.040136e+09             1.0
1     IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01                       29                   1.615563e+09         3.000000e-01             1.0
2     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00                       29                   1.615563e+09         3.000000e+00             1.0
3     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00                       28                   1.615563e+09         1.000000e+00             1.0
4     IG02          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01                       19                   1.615563e+09         5.800000e+01             1.0
5     IG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01                       16                   1.615563e+09         9.800000e+01             1.0
6     IG02    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent  1.636620e+01                       10                   1.615563e+09         1.636620e+01             1.0
7     IG02        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity  1.541700e+01                       10                   1.615563e+09         1.541700e+01             1.0
8     IG02  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...  4.715000e-01                       10                   1.615563e+09         4.715000e-01             1.0
9     IG02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...  4.200000e-03                       10                   1.615563e+09         4.200000e-03             1.0
10    IG02  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...  1.590000e-02                       10                   1.615563e+09         1.590000e-02             1.0
11    IG02      OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace  9.950000e+03                        9                   1.615563e+09         9.950000e+03             1.0
12    IG02           OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace  4.320000e+03                        9                   1.615563e+09         4.320000e+03             1.0
13    IG02  OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace  7.680000e+02                        9                   1.615563e+09         7.680000e+02             1.0
14    IG02       OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace  1.400000e+02                        9                   1.615563e+09         1.400000e+02             1.0
15    IG02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvail...  3.730000e+03                        9                   1.615563e+09         3.730000e+03             1.0
16    IG02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSUsedS...  1.500000e+01                        9                   1.615563e+09         1.500000e+01             1.0
17    IG02  OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace  9.202000e+03                        9                   1.615563e+09         9.202000e+03             1.0
18    IG02       OSLinux-OSLinux_FILESYSTEM_-home_FSUsedSpace  2.840000e+02                        9                   1.615563e+09         2.840000e+02             1.0
19    IG02   OSLinux-OSLinux_FILESYSTEM_-tmp_FSAvailableSpace  4.657000e+03                        9                   1.615563e+09         4.657000e+03             1.0,     timestamp cmdb_id                                           kpi_name         value
0  1615563000    IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  1615563000    IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00
2  1615563000    IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01
3  1615563000    IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00
4  1615563060    IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Ensure timezone (UTC+8) variable
tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure correct dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
if 'value' in metric_df.columns:
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Filter rows for IG02 (do NOT filter by time for global thresholds)
ig02_df = metric_df[metric_df['cmdb_id'] == 'IG02'].copy()

# Overall min/max timestamp across IG02
if not ig02_df.empty and ig02_df['timestamp'].notna().any():
    ig02_overall_ts = {
        'overall_min_timestamp': int(ig02_df['timestamp'].min()),
        'overall_max_timestamp': int(ig02_df['timestamp'].max())
    }
else:
    ig02_overall_ts = {'overall_min_timestamp': pd.NA, 'overall_max_timestamp': pd.NA}
ig02_overall_ts_df = pd.DataFrame([ig02_overall_ts])

# Per (IG02, kpi_name) aggregates using the full IG02 series
if not ig02_df.empty:
    agg_basic = ig02_df.groupby(['cmdb_id', 'kpi_name']).agg(
        count=('timestamp', 'size'),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        max_value=('value', 'max')
    ).reset_index()

    # Global P50 and P95 (quantiles computed on the entire KPI series for IG02)
    q = ig02_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile([0.5, 0.95]).unstack(level=-1)
    q = q.rename(columns={0.5: 'p50', 0.95: 'p95'}).reset_index()

    # Merge aggregates and quantiles
    agg_ig02 = agg_basic.merge(q, on=['cmdb_id', 'kpi_name'], how='left')

    # Select and order columns compactly
    agg_ig02 = agg_ig02[['cmdb_id', 'kpi_name', 'count', 'min_ts', 'max_ts', 'p50', 'p95', 'max_value']]

    # Sort by count desc and limit to 50 rows
    agg_ig02_top50 = agg_ig02.sort_values('count', ascending=False).head(50).reset_index(drop=True)
else:
    agg_ig02_top50 = pd.DataFrame(columns=['cmdb_id','kpi_name','count','min_ts','max_ts','p50','p95','max_value'])

# Final compact outputs (DataFrame and overall timestamps)
agg_ig02_top50, ig02_overall_ts_df

```
Out[7]:
```
```python
# Summarize IG02 KPI aggregates computed earlier and provide a plain-English summary string.
# Reuse kernel variables: agg_ig02_top50, ig02_overall_ts_df

try:
    df = agg_ig02_top50.copy()
    overall = ig02_overall_ts_df.iloc[0].to_dict()
except NameError:
    raise RuntimeError("Expected aggregation variables 'agg_ig02_top50' and 'ig02_overall_ts_df' not found in kernel.")

# Compose summary
parts = []
parts.append("IG02 KPI series global aggregates (full available series):")

# overall timeframe
omin = overall.get('overall_min_timestamp', 'N/A')
omax = overall.get('overall_max_timestamp', 'N/A')
parts.append(f"- Overall timestamp range for IG02: {omin} to {omax} (epoch seconds).")

# number of series (reported up to 50 rows returned)
n_series = df.shape[0]
parts.append(f"- Returning up to {n_series} series (sorted by count desc; limited to 50).")

# Note about typical count
if n_series > 0:
    common_count = int(df['count'].mode().iloc[0]) if not df['count'].mode().empty else None
    if common_count:
        parts.append(f"- Many top series have count = {common_count} (per-minute coverage across the observed interval).")

# Examples: show top 5 series details
parts.append("- Example top series (kpi_name — count, global P50, global P95, max):")
for _, r in df.head(5).iterrows():
    k = r['kpi_name']
    c = int(r['count'])
    p50 = r['p50']
    p95 = r['p95']
    mx = r['max_value']
    parts.append(f"  - {k} — count={c}, p50={p50:.6g}, p95={p95:.6g}, max={mx:.6g}")

# High-level interpretation
parts.append("- Interpretation: IG02 shows many KPIs with full-series coverage (counts around 30 for top rows).")
parts.append("  Several system and JVM metrics (e.g., JVM heap max/used, thread counts, memory usage) have p95 values reached in the data; max values often equal the p95 for top series, indicating values met but did not substantially exceed the P95 threshold during the observed interval.")

summary = " ".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  count      min_ts      max_ts           p50           p95     max_value
0     IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     30  1615563000  1615564740  2.040136e+09  2.040136e+09  2.040136e+09
1     IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     30  1615563000  1615564740  5.094843e+08  8.513184e+08  8.811736e+08
2     IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     30  1615563000  1615564740  5.781613e+07  5.781925e+07  5.781925e+07
3     IG02           JVM-Threads_7778_JVM_ThreadCount_Threads     30  1615563000  1615564740  7.200000e+01  7.600000e+01  7.600000e+01
4     IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     30  1615563000  1615564740  0.000000e+00  0.000000e+00  0.000000e+00
..     ...                                                ...    ...         ...         ...           ...           ...           ...
45    IG02      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity     15  1615563000  1615564680  2.852800e+01  2.855288e+01  2.855610e+01
46    IG02  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsed...     15  1615563000  1615564680  9.658800e-02  9.660570e-02  9.661900e-02
47    IG02  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...     15  1615563000  1615564680  1.590000e-02  1.590000e-02  1.590000e-02
48    IG02        OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity     15  1615563000  1615564680  3.001308e+00  3.001432e+00  3.001432e+00
49    IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps     15  1615563000  1615564680  8.000000e-01  1.091690e+00  1.150000e+00

[50 rows x 8 columns],    overall_min_timestamp  overall_max_timestamp
0             1615563000             1615564740)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Ensure timezone (UTC+8) variable exists
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp numeric
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')

# Filter for IG02 and aggregate per kpi_name using full series
ig02_df = metric_df[metric_df['cmdb_id'] == 'IG02'].copy()

ig02_kpi_summary = ig02_df.groupby('kpi_name', as_index=False).agg(
    count=('timestamp', 'size'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max')
)

# Sort by count desc and limit to top 200
ig02_kpi_summary_top200 = ig02_kpi_summary.sort_values('count', ascending=False).head(200).reset_index(drop=True)

# Display the compact table with requested columns
ig02_kpi_summary_top200[['kpi_name', 'count', 'min_ts', 'max_ts']]

```
Out[8]:
```
```python
# Summarize IG02 KPI list and guidance for inspection around epoch 1615563720.
# Reuse existing kernel variable: ig02_kpi_summary_top200

import pandas as pd
import pytz

try:
    df = ig02_kpi_summary_top200.copy()
except NameError:
    raise RuntimeError("Expected 'ig02_kpi_summary_top200' not found in kernel.")

# basic counts and timeframe
total_kpis = int(df.shape[0])
full_coverage_count = int((df['count'] == df['count'].max()).sum())  # how many have the top count (typically 30)
overall_min = int(df['min_ts'].min()) if not df['min_ts'].isna().all() else 'N/A'
overall_max = int(df['max_ts'].max()) if not df['max_ts'].isna().all() else 'N/A'

# top example KPI names
top_examples = df.head(8)[['kpi_name','count']].to_dict(orient='records')

# Compose plain-English summary
summary_lines = [
    f"IG02 KPI inventory (using full available series):",
    f"- Distinct KPI series found: {total_kpis}.",
    f"- Time range covered by IG02 metrics: {overall_min} to {overall_max} (epoch seconds).",
    f"- {full_coverage_count} series have the maximum sample count (top coverage, e.g., 30).",
    "",
    "Top example KPIs (name — count):"
]
for rec in top_examples:
    summary_lines.append(f"  - {rec['kpi_name']} — {int(rec['count'])}")

summary_lines += [
    "",
    "Recommendation for inspection around epoch 1615563720:",
    "- Prioritize JVM and memory KPIs (e.g., HeapMemoryMax, HeapMemoryUsed, NoHeapMemoryUsed, ThreadCount).",
    "- Check OS/process KPIs (process counts, zabbix agent process, per-process memory CPU metrics).",
    "- Inspect network metrics shown (TCP CLOSE-WAIT / FIN-WAIT) for socket issues.",
    "- Review filesystem/disk KPIs (FSUsed/FSAvailable, localdisk DSK* metrics) and disk I/O.",
    "- Also consider CPU and swap series if present; these KPI categories are included in the list and often carry full-minute coverage.",
    "",
    "Actionable next step: pick relevant KPI names from the returned table 'ig02_kpi_summary_top200' (kpi_name column) to drill into per-minute values around 1615563720."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  count      min_ts      max_ts
0            JVM-Memory_7778_JVM_Memory_HeapMemoryMax     30  1615563000  1615564740
1           JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     30  1615563000  1615564740
2         JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     30  1615563000  1615564740
3            JVM-Threads_7778_JVM_ThreadCount_Threads     30  1615563000  1615564740
4      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     30  1615563000  1615564740
..                                                ...    ...         ...         ...
84  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...      6  1615563000  1615564560
85       OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname      3  1615563000  1615564260
86   OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute      3  1615563360  1615564560
87        OSLinux-OSLinux_FILE_-home-zabbix_DirSizeMB      2  1615563360  1615564560
88           OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapSize      1  1615563900  1615563900

[89 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df from kernel (already loaded in previous steps)
# Ensure types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Requested KPIs for IG02
requested_kpis = [
    "JVM-Memory_7778_JVM_Memory_HeapMemoryUsed",
    "JVM-Threads_7778_JVM_ThreadCount_Threads",
    "OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc",
    "OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT",
    "OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps"
]

# 1) Compute global P95 for each requested (IG02, kpi_name) using the full series BEFORE window filtering
ig02_full = metric_df[metric_df['cmdb_id'] == 'IG02'].copy()

# compute global p95 for requested KPIs (ensures rule 9)
gp95_all = ig02_full[ig02_full['kpi_name'].isin(requested_kpis)].groupby(
    ['cmdb_id','kpi_name'], as_index=False
)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# If any requested KPI missing from gp95_all (no full-series data), include them with NaN gp95
missing = [k for k in requested_kpis if k not in gp95_all['kpi_name'].values]
if missing:
    miss_df = pd.DataFrame([{'cmdb_id':'IG02','kpi_name':k,'global_p95':np.nan} for k in missing])
    gp95_all = pd.concat([gp95_all, miss_df], ignore_index=True)

# 2) Define incident window (UTC+8) and filter per-minute rows for IG02 and requested KPIs
start_ts = int(pd.Timestamp('2021-03-12 23:30:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-13 00:00:00', tz=tz).timestamp())

window_rows = ig02_full[
    (ig02_full['timestamp'] >= start_ts) &
    (ig02_full['timestamp'] <= end_ts) &
    (ig02_full['kpi_name'].isin(requested_kpis))
].copy()

# Attach global_p95 to each window row
window_with_gp95 = window_rows.merge(gp95_all, on=['cmdb_id','kpi_name'], how='left')

# 3) Determine anomaly condition per prior convention:
#    - if gp95 is NaN -> cannot evaluate -> no anomaly
#    - if gp95 == 0 -> anomaly if value != 0
#    - else -> anomaly if value >= gp95
def is_anom(row):
    gp = row['global_p95']
    v = row['value']
    if pd.isna(gp) or pd.isna(v):
        return False
    if gp == 0:
        return v != 0
    return v >= gp

window_with_gp95['is_anomaly'] = window_with_gp95.apply(is_anom, axis=1)

# 4) Prepare per-minute sample rows (limit to at most 100 rows) sorted by kpi_name then timestamp
sample_rows_df = window_with_gp95[['timestamp','cmdb_id','kpi_name','value','global_p95']].sort_values(
    ['kpi_name','timestamp']
).head(100).reset_index(drop=True)

# 5) Aggregate per KPI summary as requested
summary_rows = []
for kpi in requested_kpis:
    subset = window_with_gp95[window_with_gp95['kpi_name'] == kpi]
    gp_row = gp95_all[(gp95_all['kpi_name']==kpi) & (gp95_all['cmdb_id']=='IG02')]
    gp95 = float(gp_row['global_p95'].iloc[0]) if not gp_row.empty and not pd.isna(gp_row['global_p95'].iloc[0]) else np.nan
    if subset.empty:
        an_count = 0
        earliest_ts = np.nan
        maxv = np.nan
    else:
        an_count = int(subset['is_anomaly'].sum())
        earliest_ts = int(subset.loc[subset['is_anomaly'],'timestamp'].min()) if an_count > 0 else np.nan
        maxv = float(subset['value'].max()) if not subset['value'].isna().all() else np.nan
    # severity ratio handling
    if pd.isna(gp95):
        severity = np.nan
    else:
        if gp95 == 0:
            if pd.isna(maxv) or maxv == 0:
                severity = np.nan
            else:
                severity = np.inf
        else:
            severity = maxv / gp95 if not pd.isna(maxv) else np.nan
    summary_rows.append({
        'cmdb_id': 'IG02',
        'kpi_name': kpi,
        'global_p95': gp95,
        'anomaly_count_in_window': an_count,
        'earliest_anomaly_ts_in_window': earliest_ts,
        'max_value_in_window': maxv,
        'severity_ratio': severity
    })

summary_df = pd.DataFrame(summary_rows)

# Keep full intermediate variables in kernel memory:
# gp95_all, window_with_gp95, window_rows, ig02_full

# Final compact outputs (per-minute sample rows limited to 100, and per-KPI summary for the requested KPIs)
sample_rows_df, summary_df

```
Out[9]:
```
```python
# Build a plain-English summary string from the kernel results (sample_rows_df, summary_df)
import pandas as pd, math

# reuse existing variables from kernel
try:
    sample = sample_rows_df.copy()
    summ = summary_df.copy()
except NameError:
    raise RuntimeError("Expected 'sample_rows_df' and 'summary_df' in kernel.")

def fmt_num(x):
    if pd.isna(x):
        return "N/A"
    if isinstance(x, (float,int)):
        if math.isinf(x):
            return "inf"
        # show integers without decimal when large whole numbers (e.g., bytes), else 3 significant digits
        if abs(x) >= 1e6 and float(x).is_integer():
            return f"{int(x)}"
        return f"{float(x):.6g}"
    return str(x)

lines = []
lines.append("Summary of requested IG02 KPIs inside incident window (2021-03-12 23:30 to 2021-03-13 00:00 UTC+8):")
lines.append(f"- Per-minute sample rows returned (capped): {len(sample)} rows (showing up to 100).")

# Summarize each KPI row in summary_df
lines.append("- Per-KPI findings (global_p95, anomaly_count_in_window, earliest_anomaly_ts_in_window, max_value_in_window, severity_ratio):")
for _, r in summ.iterrows():
    k = r['kpi_name']
    gp = fmt_num(r['global_p95'])
    cnt = int(r['anomaly_count_in_window'])
    earliest = int(r['earliest_anomaly_ts_in_window']) if not pd.isna(r['earliest_anomaly_ts_in_window']) else "N/A"
    maxv = fmt_num(r['max_value_in_window'])
    sev = fmt_num(r['severity_ratio'])
    lines.append(f"  - {k} — gp95={gp}, anomalies={cnt}, earliest={earliest}, max={maxv}, severity={sev}")

# Short interpretation & recommendations
lines.append("")
lines.append("Key observations:")
# heap memory
heap = summ[summ['kpi_name'].str.contains("HeapMemoryUsed", na=False)]
if not heap.empty:
    h = heap.iloc[0]
    lines.append(f"- JVM heap (HeapMemoryUsed) exceeded its global P95 in the window (anomaly_count={int(h['anomaly_count_in_window'])}) with max > gp95 (severity ~{fmt_num(h['severity_ratio'])}). Inspect JVM heap around those timestamps.")
# threads
thr = summ[summ['kpi_name'].str.contains("ThreadCount_Threads", na=False)]
if not thr.empty:
    t = thr.iloc[0]
    lines.append(f"- Thread count reached P95 (count={int(t['anomaly_count_in_window'])}), severity={fmt_num(t['severity_ratio'])}.")
# memory percent
memp = summ[summ['kpi_name'].str.contains("MEMUsedMemPerc", na=False)]
if not memp.empty:
    m = memp.iloc[0]
    lines.append(f"- Memory percentage (MEMUsedMemPerc) hit P95 frequently (anomaly_count={int(m['anomaly_count_in_window'])}); severity={fmt_num(m['severity_ratio'])}.")
# tcp close-wait
tcp = summ[summ['kpi_name'].str.contains("TCP-CLOSE-WAIT", na=False)]
if not tcp.empty:
    tc = tcp.iloc[0]
    if tc['global_p95'] == 0:
        lines.append("- TCP CLOSE-WAIT global P95 == 0; anomalies defined as non-zero values. No anomalies detected in the window (values are 0).")
    else:
        lines.append(f"- TCP CLOSE-WAIT anomalies: count={int(tc['anomaly_count_in_window'])}, severity={fmt_num(tc['severity_ratio'])}.")
# proc mem
ppmem = summ[summ['kpi_name'].str.contains("PROCPPMem", na=False)]
if not ppmem.empty:
    p = ppmem.iloc[0]
    lines.append(f"- Per-process memory (PROCESS_PROCPPMem) reached P95 often (anomaly_count={int(p['anomaly_count_in_window'])}); severity={fmt_num(p['severity_ratio'])}.")
# disk write TPS
dsk = summ[summ['kpi_name'].str.contains("DSKWTps", na=False)]
if not dsk.empty:
    d = dsk.iloc[0]
    lines.append(f"- Disk write TPS (sda DSKWTps) had {int(d['anomaly_count_in_window'])} anomaly point(s) with severity ~{fmt_num(d['severity_ratio'])}; inspect I/O around those timestamps.")

lines.append("")
lines.append("Recommended next steps:")
lines.append("- Focus inspection on JVM heap (HeapMemoryUsed) and disk write TPS (sda DSKWTps) because they exceeded global P95 (severity > 1).")
lines.append("- Also review memory percentage and per-process memory (PROCPPMem) which reached P95 frequently (severity ~1.0).")
lines.append("- No TCP CLOSE-WAIT issues detected in this window (values are zero).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp cmdb_id                                        kpi_name        value   global_p95
0   1615563000    IG02       JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  321428280.0  851318422.4
1   1615563060    IG02       JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  361091584.0  851318422.4
2   1615563120    IG02       JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  397591616.0  851318422.4
3   1615563180    IG02       JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  433444816.0  851318422.4
4   1615563240    IG02       JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  468319144.0  851318422.4
..         ...     ...                                             ...          ...          ...
95  1615563000    IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          0.0          0.0
96  1615563060    IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          0.0          0.0
97  1615563120    IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          0.0          0.0
98  1615563180    IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          0.0          0.0
99  1615563240    IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          0.0          0.0

[100 rows x 5 columns],   cmdb_id                                         kpi_name    global_p95  anomaly_count_in_window  earliest_anomaly_ts_in_window  max_value_in_window  severity_ratio
0    IG02        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.513184e+08                        2                   1.615564e+09         8.811736e+08        1.035069
1    IG02         JVM-Threads_7778_JVM_ThreadCount_Threads  7.600000e+01                        3                   1.615563e+09         7.600000e+01        1.000000
2    IG02     OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01                       16                   1.615563e+09         9.800000e+01        1.000000
3    IG02   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00                        0                            NaN         0.000000e+00             NaN
4    IG02        OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01                       19                   1.615563e+09         5.800000e+01        1.000000
5    IG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps  1.091690e+00                        1                   1.615564e+09         1.150000e+00        1.053413)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames if present, else load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv('error_logs.csv')

# Ensure types
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
if 'value' in log_df.columns:
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce').astype('Int64')

# Requested cmdb_ids and window
requested_cmdbs = ['IG02','IG01','Tomcat01','Tomcat02','Tomcat03','Tomcat04','apache01','apache02']
start_ts = 1615563000
end_ts = 1615564740

# Filter log.csv to window and requested cmdbs
log_win = log_df[
    (log_df['cmdb_id'].isin(requested_cmdbs)) &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

# 1) Aggregation per (cmdb_id, log_name): count, min_ts, max_ts, p50, p95, max(value)
if not log_win.empty:
    agg_basic = log_win.groupby(['cmdb_id','log_name'], as_index=False).agg(
        count=('timestamp','size'),
        min_ts=('timestamp','min'),
        max_ts=('timestamp','max'),
        max_value=('value','max')
    )
    # quantiles
    q = log_win.groupby(['cmdb_id','log_name'])['value'].quantile([0.5,0.95]).unstack(level=-1).reset_index()
    q = q.rename(columns={0.5: 'p50', 0.95: 'p95'})
    log_agg = agg_basic.merge(q, on=['cmdb_id','log_name'], how='left')
    log_agg_top20 = log_agg.sort_values('count', ascending=False).head(20).reset_index(drop=True)
else:
    log_agg_top20 = pd.DataFrame(columns=['cmdb_id','log_name','count','min_ts','max_ts','max_value','p50','p95'])

# 2) Up to 20 sample rows from log_win ordered by timestamp asc
log_sample_rows = log_win.sort_values('timestamp')[['timestamp','cmdb_id','log_name','value']].head(20).reset_index(drop=True)

# 3) error_logs.csv: filter to requested cmdbs and window
err_win = error_logs_df[
    (error_logs_df['cmdb_id'].isin(requested_cmdbs)) &
    (error_logs_df['timestamp'] >= start_ts) &
    (error_logs_df['timestamp'] <= end_ts)
].copy()

if not err_win.empty:
    error_agg = err_win.groupby('cmdb_id', as_index=False).agg(
        count=('timestamp','size'),
        min_ts=('timestamp','min'),
        max_ts=('timestamp','max')
    ).sort_values('count', ascending=False).reset_index(drop=True)
    # up to 10 earliest sample messages
    error_sample_messages = err_win.sort_values('timestamp')[['timestamp','cmdb_id','message']].head(10).reset_index(drop=True)
else:
    error_agg = pd.DataFrame(columns=['cmdb_id','count','min_ts','max_ts'])
    error_sample_messages = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Keep intermediates in variables: log_win, log_agg, err_win

# Final compact outputs
log_agg_top20, log_sample_rows, error_agg, error_sample_messages

```
Out[10]:
```
```python
# Build a concise plain-English summary from previous results in kernel:
# Variables assumed present: log_agg_top20, log_sample_rows, error_agg, error_sample_messages

parts = []
parts.append("Log scan summary for window 1615563000–1615564740 (UTC+8) and cmdb_ids IG02, IG01, Tomcat01-04, apache01-02:")

# Aggregation highlights
parts.append("1) log.csv aggregates (top results):")
parts.append("- Most common log series (count=26) are per-node log.row_count and log.error_count for Tomcat and Apache.")
parts.append("- log.error_count values are 0 in the window for these nodes (no error spikes in aggregated counts).")
parts.append("- log.row_count shows typical per-minute row volumes (examples: Tomcat01 row_count p50=72.5, p95=90; apache01 p50=146, p95=173).")
parts.append("- Some GC metrics present (log.gc_pause_max) for Tomcat nodes with small sample counts (4–7) and p95 ~0.08–0.12s.")

# Sample rows note
parts.append("2) Sample log rows (earliest entries):")
parts.append("- Earliest per-minute samples show normal baseline values (e.g., row_count entries at 1615563000; error_count = 0).")

# Error logs
if (('error_agg' in globals() and not error_agg.empty) if 'error_agg' in globals() else False):
    parts.append("3) error_logs: some entries present (see aggregated counts).")
else:
    parts.append("3) error_logs: no error_log entries found for the requested components in the window.")

# Short interpretation
parts.append("Interpretation: No aggregate log.error spikes or error_log messages were found in the window for these components. Operational metrics like row_count and occasional GC pauses are present, but nothing in logs indicates elevated error activity during this incident window. Investigate metrics (CPU, JVM heap, traces) rather than application log errors based on these results.")

summary = " ".join(parts)

# Display concise summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  count      min_ts      max_ts  max_value      p50       p95
0   Tomcat01   log.error_count     26  1615563000  1615564500       0.00    0.000    0.0000
1   Tomcat01     log.row_count     26  1615563000  1615564500     109.00   72.500   90.0000
2   Tomcat02   log.error_count     26  1615563000  1615564500       0.00    0.000    0.0000
3   Tomcat02     log.row_count     26  1615563000  1615564500     102.00   71.500   95.2500
4   Tomcat03   log.error_count     26  1615563000  1615564500       0.00    0.000    0.0000
5   Tomcat04     log.row_count     26  1615563000  1615564500      98.00   71.000   93.5000
6   Tomcat04   log.error_count     26  1615563000  1615564500       0.00    0.000    0.0000
7   Tomcat03     log.row_count     26  1615563000  1615564500      93.00   72.000   89.5000
8   apache02   log.error_count     26  1615563000  1615564500       0.00    0.000    0.0000
9   apache01     log.row_count     26  1615563000  1615564500     183.00  146.000  173.7500
10  apache02     log.row_count     26  1615563000  1615564500     190.00  146.500  178.0000
11  apache01   log.error_count     26  1615563000  1615564500       0.00    0.000    0.0000
12  Tomcat02  log.gc_pause_max      7  1615563000  1615564380       0.09    0.060    0.0840
13  Tomcat03  log.gc_pause_max      5  1615563000  1615564320       0.13    0.050    0.1160
14  Tomcat01  log.gc_pause_max      4  1615563240  1615564320       0.08    0.065    0.0785
15  Tomcat04  log.gc_pause_max      4  1615563180  1615564320       0.12    0.075    0.1155,      timestamp   cmdb_id          log_name   value
0   1615563000  Tomcat01   log.error_count    0.00
1   1615563000  Tomcat01     log.row_count   63.00
2   1615563000  Tomcat02   log.error_count    0.00
3   1615563000  Tomcat02  log.gc_pause_max    0.04
4   1615563000  Tomcat02     log.row_count   63.00
5   1615563000  Tomcat03   log.error_count    0.00
6   1615563000  Tomcat03  log.gc_pause_max    0.02
7   1615563000  Tomcat03     log.row_count   79.00
8   1615563000  Tomcat04   log.error_count    0.00
9   1615563000  Tomcat04     log.row_count   61.00
10  1615563000  apache01   log.error_count    0.00
11  1615563000  apache01     log.row_count  130.00
12  1615563000  apache02   log.error_count    0.00
13  1615563000  apache02     log.row_count  134.00
14  1615563060  apache02     log.row_count  109.00
15  1615563060  apache02   log.error_count    0.00
16  1615563060  Tomcat02     log.row_count   60.00
17  1615563060  Tomcat02   log.error_count    0.00
18  1615563060  apache01     log.row_count  115.00
19  1615563060  apache01   log.error_count    0.00, Empty DataFrame
Columns: [cmdb_id, count, min_ts, max_ts]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import re

# ensure metric_df and ig02_df exist (reuse if available)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Filter IG02 full-series (no time filtering for global aggregates)
ig02_df = metric_df[metric_df['cmdb_id'] == 'IG02'].copy()

# Keywords to match (case-insensitive). Build regex pattern.
keywords = ['CPU','Mem','Memory','Heap','Thread','GC','DSK','LOCALDISK','NETWORK','TCP','PROCESS','SWAP','FILESYSTEM']
pattern = '|'.join(keywords)

# Find KPI names matching any keyword (case-insensitive)
mask = ig02_df['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
ig02_kpi_matches = ig02_df[mask].copy()

# Aggregate per kpi_name using full series
ig02_kpi_summary = ig02_kpi_matches.groupby('kpi_name', as_index=False).agg(
    count=('timestamp','size'),
    min_ts=('timestamp','min'),
    max_ts=('timestamp','max')
)

# Sort by count desc and limit to top 200
ig02_kpi_matches_top200 = ig02_kpi_summary.sort_values('count', ascending=False).head(200).reset_index(drop=True)

# Return only requested columns compactly
ig02_kpi_matches_top200[['kpi_name','count','min_ts','max_ts']]

```
Out[11]:
```
```python
# Build a concise plain-English summary of the IG02 KPI matches (resource-related KPIs).
# Reuse kernel variable ig02_kpi_matches_top200 if available; otherwise recompute from metric_df.

import pandas as pd, pytz

try:
    df = ig02_kpi_matches_top200.copy()
except NameError:
    # fallback: recompute quickly
    try:
        metric_df
    except NameError:
        metric_df = pd.read_csv('metric.csv')
    ig02_df = metric_df[metric_df['cmdb_id']=='IG02'].copy()
    keywords = ['CPU','Mem','Memory','Heap','Thread','GC','DSK','LOCALDISK','NETWORK','TCP','PROCESS','SWAP','FILESYSTEM']
    pattern = '|'.join(keywords)
    mask = ig02_df['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
    df = ig02_df[mask].groupby('kpi_name', as_index=False).agg(
        count=('timestamp','size'),
        min_ts=('timestamp','min'),
        max_ts=('timestamp','max')
    ).sort_values('count', ascending=False).head(200).reset_index(drop=True)

# Compose summary
total_kpis = int(df.shape[0])
top_count = int(df['count'].max()) if total_kpis>0 else 0
num_full_coverage = int((df['count'] == top_count).sum()) if total_kpis>0 else 0
examples = df.head(8)[['kpi_name','count']].to_dict(orient='records')

lines = []
lines.append(f"IG02 resource-related KPI candidates matching keywords (total matched: {total_kpis}).")
lines.append(f"- Timezone used conceptually: UTC+8. Returned up to top 200 KPI names (sorted by count).")
lines.append(f"- {num_full_coverage} series have the highest sample count (count={top_count}), i.e., full-minute coverage over the observed period.")
lines.append("- Example top KPI names (name — count):")
for rec in examples:
    lines.append(f"  - {rec['kpi_name']} — {int(rec['count'])}")
lines.append("")
lines.append("Recommended KPIs to inspect around epoch 1615563720:")
lines.append("- JVM memory: HeapMemoryUsed, HeapMemoryMax, NoHeapMemoryUsed, JVM thread counts.")
lines.append("- Memory percent (MEMUsedMemPerc) and per-process memory (PROCESS_PROCPPMem).")
lines.append("- CPU series (SingleCpuUtil / CPUidle / CPULoad) and swap if present.")
lines.append("- Disk I/O and capacity: LOCALDISK DSK* metrics, filesystem FSUsed/FSAvailable.")
lines.append("- Network / TCP series (e.g., TCP-CLOSE-WAIT, TCP-FIN-WAIT) for socket issues.")
lines.append("")
lines.append("Action: use the returned 'kpi_name' list to pick specific KPIs for per-minute inspection at/around 1615563720.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  count      min_ts      max_ts
0            JVM-Memory_7778_JVM_Memory_HeapMemoryMax     30  1615563000  1615564740
1           JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     30  1615563000  1615564740
2         JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     30  1615563000  1615564740
3            JVM-Threads_7778_JVM_ThreadCount_Threads     30  1615563000  1615564740
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     30  1615563000  1615564740
..                                                ...    ...         ...         ...
78                    OSLinux-CPU_CPU-2_SingleCpuUtil      8  1615563180  1615564620
79                    OSLinux-CPU_CPU-0_SingleCpuUtil      8  1615563180  1615564620
80  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...      6  1615563000  1615564560
81  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...      6  1615563000  1615564560
82           OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapSize      1  1615563900  1615563900

[83 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

# reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# ensure correct dtypes
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Target component and trace names
cmdb_target = 'IG02'
trace_names = [
    'trace.self.row_count',
    'trace.self.duration_mean',
    'trace.self.duration_p95'
]

# Incident window (epoch seconds, UTC+8)
start_ts = 1615563000
end_ts   = 1615564740

# 1) Compute global P95 for each target series using the full trace_df (global BEFORE filtering)
# Reuse global_p95_trace_df if available
try:
    global_p95_trace_df
except NameError:
    global_p95_trace_df = trace_df.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# Ensure we have gp95 rows for the requested series; compute any missing explicitly
gp95_needed = pd.DataFrame([(cmdb_target, tn) for tn in trace_names], columns=['cmdb_id','trace_name'])
gp95_targets = gp95_needed.merge(global_p95_trace_df, on=['cmdb_id','trace_name'], how='left')

missing_mask = gp95_targets['global_p95'].isna()
if missing_mask.any():
    missing_pairs = gp95_targets[missing_mask][['cmdb_id','trace_name']]
    if not missing_pairs.empty:
        comp = trace_df.merge(missing_pairs, on=['cmdb_id','trace_name'], how='inner')
        if not comp.empty:
            comp_gp95 = comp.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})
            gp95_targets = gp95_targets.drop(columns=['global_p95']).merge(comp_gp95, on=['cmdb_id','trace_name'], how='left')

# 2) Extract per-minute rows for IG02 within window for the three trace_names
window_trace_rows = trace_df[
    (trace_df['cmdb_id'] == cmdb_target) &
    (trace_df['trace_name'].isin(trace_names)) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# keep only requested columns
window_trace_rows = window_trace_rows[['timestamp','cmdb_id','trace_name','value']].sort_values(['timestamp','trace_name']).reset_index(drop=True)

# 3) Attach global_p95 to window rows for convenience
window_with_gp95 = window_trace_rows.merge(gp95_targets, on=['cmdb_id','trace_name'], how='left')

# Define anomaly detection rule:
# - if global_p95 is NaN -> no anomaly
# - if global_p95 == 0 -> anomaly if value != 0
# - else -> anomaly if value >= global_p95
def is_anomaly_val(v, gp):
    if pd.isna(gp) or pd.isna(v):
        return False
    if gp == 0:
        return v != 0
    return v >= gp

window_with_gp95['is_anomaly'] = window_with_gp95.apply(lambda r: is_anomaly_val(r['value'], r['global_p95']), axis=1)

# 4) Per-series summary: earliest timestamp in window where value >= gp95, max value in window, severity_ratio
summary_rows = []
for _, row in gp95_targets.iterrows():
    cmdb = row['cmdb_id']
    tname = row['trace_name']
    gp95 = row['global_p95']
    subset = window_with_gp95[(window_with_gp95['cmdb_id']==cmdb) & (window_with_gp95['trace_name']==tname)]
    if subset.empty:
        earliest = np.nan
        maxv = np.nan
    else:
        anomalies = subset[subset['is_anomaly']]
        earliest = int(anomalies['timestamp'].min()) if not anomalies.empty else np.nan
        maxv = float(subset['value'].max()) if not subset['value'].isna().all() else np.nan
    # severity ratio handling
    if pd.isna(gp95):
        severity = np.nan
    else:
        if gp95 == 0:
            if pd.isna(maxv) or maxv == 0:
                severity = np.nan
            else:
                severity = np.inf
        else:
            severity = maxv / gp95 if not pd.isna(maxv) else np.nan
    summary_rows.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'global_p95': gp95,
        'earliest_anomaly_ts_in_window': earliest,
        'max_value_in_window': maxv,
        'severity_ratio': severity
    })

summary_df = pd.DataFrame(summary_rows)[['cmdb_id','trace_name','global_p95','earliest_anomaly_ts_in_window','max_value_in_window','severity_ratio']]

# 5) Prepare up to 30 per-minute rows combining the three series ordered by timestamp asc
sample_rows_df = window_with_gp95[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').head(30).reset_index(drop=True)

# Keep intermediate variables in kernel: global_p95_trace_df, gp95_targets, window_trace_rows, window_with_gp95

# Final compact outputs
summary_df, sample_rows_df

```
Out[12]:
```
```python
# Build a concise plain-English summary from the trace analysis results in the kernel.
# Reuse existing variables: summary_df, sample_rows_df

import math
import pandas as pd
import numpy as np

# Access summary_df and sample_rows_df (computed earlier)
sdf = summary_df.copy()
samples = sample_rows_df.copy()

# Helper to format numbers
def fmt(x, sig=6):
    if pd.isna(x):
        return "N/A"
    if isinstance(x, (float, np.floating, int, np.integer)):
        if math.isinf(x):
            return "inf"
        # show integer-like large numbers without decimals
        xv = float(x)
        if abs(xv) >= 1e6 and xv.is_integer():
            return str(int(xv))
        return f"{xv:.6g}"
    return str(x)

# Extract per-series values
def row_for(name):
    row = sdf[sdf['trace_name'] == name]
    return row.iloc[0] if not row.empty else None

r_rowcount = row_for('trace.self.row_count')
r_mean = row_for('trace.self.duration_mean')
r_p95 = row_for('trace.self.duration_p95')

lines = []
lines.append("Trace inspection summary for IG02 (window 2021-03-12 23:30–2021-03-13 00:00 UTC+8):")

# duration_mean
if r_mean is not None:
    lines.append(
        f"- trace.self.duration_mean: global_p95={fmt(r_mean['global_p95'])}; "
        f"earliest >=p95 at ts={int(r_mean['earliest_anomaly_ts_in_window']) if not pd.isna(r_mean['earliest_anomaly_ts_in_window']) else 'N/A'}; "
        f"max_in_window={fmt(r_mean['max_value_in_window'])}; severity={fmt(r_mean['severity_ratio'])}."
    )

# duration_p95
if r_p95 is not None:
    lines.append(
        f"- trace.self.duration_p95: global_p95={fmt(r_p95['global_p95'])}; "
        f"earliest >=p95 at ts={int(r_p95['earliest_anomaly_ts_in_window']) if not pd.isna(r_p95['earliest_anomaly_ts_in_window']) else 'N/A'}; "
        f"max_in_window={fmt(r_p95['max_value_in_window'])}; severity={fmt(r_p95['severity_ratio'])}."
    )

# row_count
if r_rowcount is not None:
    lines.append(
        f"- trace.self.row_count: global_p95={fmt(r_rowcount['global_p95'])}; "
        f"earliest >=p95 at ts={int(r_rowcount['earliest_anomaly_ts_in_window']) if not pd.isna(r_rowcount['earliest_anomaly_ts_in_window']) else 'N/A'}; "
        f"max_in_window={fmt(r_rowcount['max_value_in_window'])}; severity={fmt(r_rowcount['severity_ratio'])}."
    )

# Brief interpretation about ordering / leads
# Check ordering: which anomaly occurred first among the three (if any)
anoms = []
for rname in ['trace.self.duration_mean','trace.self.duration_p95','trace.self.row_count']:
    r = row_for(rname)
    if r is not None and not pd.isna(r['earliest_anomaly_ts_in_window']):
        anoms.append((rname, int(r['earliest_anomaly_ts_in_window'])))
if anoms:
    anoms_sorted = sorted(anoms, key=lambda x: x[1])
    first = anoms_sorted[0]
    lines.append(f"- Earliest detected anomaly: {first[0]} at ts={first[1]}.")
    # note about row_count increases
    lines.append("- Note: row_count shows incremental increases earlier in the window but did not exceed its global P95 until later (peak at ts shown above). Duration_mean exceeded its P95 earlier (suggesting latency spike before row_count hit peak).")

# Provide sample row guidance
lines.append("- Sample per-minute rows (up to 30) were returned for visual inspection to check if row_count increases precede duration spikes.")

# Concise recommendation
lines.append("- Recommendation: Investigate the duration_mean spike at its earliest anomaly timestamp first (check downstream Tomcat services and resource KPIs around that time), then correlate with row_count and duration_p95 anomalies.")

summary = " ".join(lines)

# Display the summary string as the final output
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                trace_name  global_p95  earliest_anomaly_ts_in_window  max_value_in_window  severity_ratio
0    IG02      trace.self.row_count  388.300000                     1615564680           606.000000        1.560649
1    IG02  trace.self.duration_mean    0.146907                     1615563720             0.343475        2.338049
2    IG02   trace.self.duration_p95    0.987752                     1615564500             1.954000        1.978228,      timestamp cmdb_id                trace_name       value
0   1615563000    IG02  trace.self.duration_mean    0.070741
1   1615563000    IG02   trace.self.duration_p95    0.356800
2   1615563000    IG02      trace.self.row_count  274.000000
3   1615563060    IG02  trace.self.duration_mean    0.044371
4   1615563060    IG02   trace.self.duration_p95    0.127000
..         ...     ...                       ...         ...
25  1615563480    IG02   trace.self.duration_p95    0.553000
26  1615563480    IG02      trace.self.row_count  286.000000
27  1615563540    IG02  trace.self.duration_mean    0.096221
28  1615563540    IG02   trace.self.duration_p95    0.539450
29  1615563540    IG02      trace.self.row_count  308.000000

[30 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```